In [ ]:
from collections import deque
import gymnasium as gym
from itertools import combinations_with_replacement
import numpy as np
import random
from sklearn.linear_model import LinearRegression, Ridge
import tqdm

In [2]:
env = gym.make("MountainCarContinuous-v0")
env.observation_space, env.action_space

(Box([-1.2  -0.07], [0.6  0.07], (2,), float32), Box(-1.0, 1.0, (1,), float32))

In [3]:
def get_predictors(x, degree):
	x = np.column_stack([np.ones(x.shape[0]), x])
	idx = list(combinations_with_replacement(range(x.shape[1]), degree))
	predictors = x[:, idx].prod(-1)
	return predictors


def model_ema(new, target, alpha):
	target.coef_ = alpha * new.coef_ + (1 - alpha) * target.coef_

In [29]:
num_episodes = 1_000
batch_size = 256
replay_size = 50_000
gamma = 0.97
alpha1 = 1.0
alpha2 = 0.1
lr = 0.02
tau = 0.005
sigma = 0.1
degree = 3

In [30]:
replay_buffer = deque(maxlen=replay_size)

critic = LinearRegression(fit_intercept=False)
actor1 = LinearRegression(fit_intercept=False)
actor2 = LinearRegression(fit_intercept=False)
target_critic = LinearRegression(fit_intercept=False)
target_actor1 = LinearRegression(fit_intercept=False)
target_actor2 = LinearRegression(fit_intercept=False)
new_critic = Ridge(alpha1, fit_intercept=False)
new_actor1 = Ridge(alpha1, fit_intercept=False)
new_actor2 = Ridge(alpha2, fit_intercept=False)

state = env.observation_space.sample().reshape(1, -1)
action = env.action_space.sample().reshape(1, -1)
state_predictors = get_predictors(state, degree)
state_action_predictors = get_predictors(np.hstack([state, action]), degree)
q_value = np.ones(1)

critic.fit(state_action_predictors, q_value)
actor1.fit(state_predictors, action)
actor2.fit(state_predictors, action)
target_critic.fit(state_action_predictors, q_value)
target_actor1.fit(state_predictors, action)
target_actor2.fit(state_predictors, action)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",False
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1, 10)","[[-0.56, 0.38,-0.02,...,-0.01, 0. ,-0. ]]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float,0
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,1
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](1,)",[1.32]


In [ ]:
reward_ema = 0
for i in tqdm.trange(num_episodes):
	state, _ = env.reset()
	done = False
	while not done:
		# Execution
		state_predictors = get_predictors(state.reshape(1, -1), degree)
		action = actor1.predict(state_predictors)[0]
		action += sigma * np.random.randn(*action.shape)
		next_state, reward, terminated, truncated, _ = env.step(action)
		reward_ema = 0.001 * reward + 0.999 * reward_ema
		done = terminated or truncated

		replay_buffer.append((state, action, reward, done, next_state))
		state = next_state

		# Training
		if len(replay_buffer) >= batch_size:
			states, actions, rewards, dones, next_states = zip(*random.sample(replay_buffer, batch_size))
			states = np.stack(states)
			actions = np.stack(actions)
			rewards = np.asarray(rewards)
			dones = np.asarray(dones)
			next_states = np.stack(next_states)

			state_predictors = get_predictors(states, degree)
			state_action_predictors = get_predictors(np.hstack([states, actions]), degree)
			next_state_predictors = get_predictors(next_states, degree)

			next_actions = target_actor1.predict(next_state_predictors)
			next_state_action_predictors = get_predictors(np.hstack([next_states, next_actions]), degree)
			next_q_values = target_critic.predict(next_state_action_predictors)
			target_q_values = rewards + gamma * (1 - dones) * next_q_values
			new_critic.fit(state_action_predictors, target_q_values)

			actions1 = actor1.predict(state_predictors)
			actions2 = actor2.predict(state_predictors)
			state_action_predictors1 = get_predictors(np.hstack([states, actions1]), degree)
			state_action_predictors2 = get_predictors(np.hstack([states, actions2]), degree)
			q_values1 = critic.predict(state_action_predictors1)
			q_values2 = critic.predict(state_action_predictors2)
			better_idx = (q_values1 > q_values2).astype(int).reshape(1, batch_size, -1)
			both_actions = np.stack([actions1, actions2])
			better_actions = np.take_along_axis(both_actions, better_idx, axis=0)[0]
			new_actor1.fit(state_predictors, better_actions)
			new_actor2.fit(state_predictors, better_actions)

			model_ema(new_critic, critic, lr)
			model_ema(new_actor1, actor1, lr)
			model_ema(new_actor2, actor2, lr)
			model_ema(critic, target_critic, tau)
			model_ema(actor1, target_actor1, tau)
			model_ema(actor2, target_actor2, tau)

	if i % 10 == 0 or i == num_episodes - 1:
		print("Reward: ", reward_ema)

  0%|          | 1/1000 [00:03<52:10,  3.13s/it]

Reward:  -0.08374586550949006


  1%|          | 11/1000 [00:44<1:07:05,  4.07s/it]

Reward:  -0.11032590392745084


  2%|▏         | 21/1000 [01:25<1:07:45,  4.15s/it]

Reward:  -0.10504585240144247


... (22 outputs left out)

 14%|█▍        | 141/1000 [10:28<1:56:28,  8.14s/it]

Reward:  -0.03107198848092737


 14%|█▍        | 141/1000 [10:32<1:04:16,  4.49s/it]


KeyboardInterrupt: 